# BIG DATA — Customer Intelligence PRO
### Single-Notebook Google Colab • Spark • SVD • CURE • FP-Growth • ALS • Executive Dashboard

Toàn bộ **luồng xử lý, code, metric, visualization và giải thích nằm trực tiếp trong notebook này**. Không gọi file Python bên ngoài.

> Mục tiêu: người xem có thể đi từ Data → Analytics → SVD → CURE → Recommendation → Spark Benchmark → Dashboard chỉ trong một file, đồng thời mọi KPI đều được tính lại từ dữ liệu thực khi chạy.

## 0. Environment

In [ ]:
# Cài các dependency cần thiết cho toàn bộ notebook.
!pip -q install pyspark==3.5.1 pyclustering==0.10.1.2 umap-learn plotly networkx psutil kaleido


## 1. Setup, data quality & Executive Dashboard shell

**Mục tiêu:** khởi tạo Spark/analytics environment, tải đúng dữ liệu từ repository, kiểm tra chất lượng dữ liệu và dựng vùng Executive Dashboard.

**Input:** `baskets.csv`, `ratings2k.csv`.  
**Output:** Spark DataFrames sạch, KPI dữ liệu, cấu hình chung cho toàn pipeline.

**Vì sao:** mọi model phía sau chỉ đáng tin khi schema, missing values, rating scale và sparsity được kiểm tra trước.


In [ ]:
# Auto-generated from BIG_DATA_Customer_Intelligence_PRO_Colab.ipynb

# ---- source cell 3 ----
import os, math, time, json, shutil, warnings, itertools, statistics
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import psutil

from IPython.display import display, HTML, clear_output
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import networkx as nx

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error,
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score,
)
import umap.umap_ as umap

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.ml.fpm import FPGrowth
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator, RankingEvaluator
from pyspark.mllib.linalg import Vectors as OldVectors
from pyspark.mllib.linalg.distributed import RowMatrix

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)
pio.templates.default = 'plotly_white'

SEED = 42
QUALITY_MODE = 'max'       # 'balanced' hoặc 'max'
TOP_K = 10
TEST_FRAC = 0.20
MIN_USER_RATINGS_FOR_TEST = 5
RELEVANCE_THRESHOLD = 4.0
STAT_SEEDS = [42, 52, 62] if QUALITY_MODE == 'max' else [42, 52]
OUTPUT_DIR = Path('/content/big_data_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

THEME = {
    'navy': '#0F172A', 'blue': '#2563EB', 'cyan': '#0891B2',
    'green': '#059669', 'amber': '#D97706', 'red': '#DC2626',
    'muted': '#64748B', 'card': '#F8FAFC'
}

spark = (
    SparkSession.builder
    .appName('BIG-DATA-Customer-Intelligence')
    .master('local[*]')
    .config('spark.driver.memory', '6g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel('WARN')
print('Spark:', spark.version, '| defaultParallelism:', sc.defaultParallelism)

# ---- source cell 5 ----
try:
    import ipywidgets as widgets
    EXEC_OUT = widgets.Output()
    display(EXEC_OUT)
    with EXEC_OUT:
        display(HTML("""
        <div style='padding:22px;border:1px solid #e2e8f0;border-radius:16px;background:#f8fafc'>
          <b>Executive Dashboard</b><br>
          <span style='color:#64748b'>Run all cells. Dashboard sẽ tự refresh khi toàn bộ pipeline hoàn tất.</span>
        </div>
        """))
except Exception:
    EXEC_OUT = None
    display(HTML('<b>Executive Dashboard:</b> sẽ được render ở phần cuối notebook.'))

# ---- source cell 7 ----
import urllib.request

BASE_RAW = 'https://raw.githubusercontent.com/NVTruong473/BIG-DATA/main/end/end/'
DATASETS = {
    'baskets.csv': BASE_RAW + 'baskets.csv',
    'ratings2k.csv': BASE_RAW + 'ratings2k.csv',
}

for name, url in DATASETS.items():
    path = Path('/content') / name
    if not path.exists() or path.stat().st_size == 0:
        print('Downloading', name)
        urllib.request.urlretrieve(url, path)
    print(name, f'{path.stat().st_size/1024:.1f} KB')

BASKETS_PATH = '/content/baskets.csv'
RATINGS_PATH = '/content/ratings2k.csv'

# ---- source cell 8 ----
baskets_raw = spark.read.option('header', True).option('inferSchema', True).csv(BASKETS_PATH)
ratings_df = (
    spark.read.option('header', True).option('inferSchema', True).csv(RATINGS_PATH)
    .select(
        F.col('user').cast('int').alias('user'),
        F.col('item').cast('int').alias('item'),
        F.col('rating').cast('double').alias('rating')
    )
    .dropna()
)

# Parse nhiều date format phổ biến để notebook robust hơn.
baskets_df = (
    baskets_raw
    .withColumn(
        'date_parsed',
        F.coalesce(
            F.to_date('Date', 'dd-MM-yyyy'),
            F.to_date('Date', 'dd/MM/yyyy'),
            F.to_date('Date', 'yyyy-MM-dd')
        )
    )
    .filter(F.col('Member_number').isNotNull() & F.col('itemDescription').isNotNull())
)

print('baskets rows =', baskets_df.count())
print('ratings rows =', ratings_df.count())
baskets_df.printSchema()
ratings_df.printSchema()

# ---- source cell 10 ----
ratings_rows = ratings_df.count()
n_users = ratings_df.select('user').distinct().count()
n_items = ratings_df.select('item').distinct().count()
rating_min, rating_max, rating_mean = ratings_df.agg(
    F.min('rating'), F.max('rating'), F.avg('rating')
).first()
sparsity = 1.0 - ratings_rows / max(1, n_users * n_items)

basket_rows = baskets_df.count()
n_members = baskets_df.select('Member_number').distinct().count()
n_baskets = baskets_df.select('Member_number', 'Date').distinct().count()
n_products = baskets_df.select('itemDescription').distinct().count()
null_dates = baskets_df.filter(F.col('date_parsed').isNull()).count()

DATA_KPIS = {
    'Rating rows': ratings_rows,
    'Users': n_users,
    'Items': n_items,
    'Rating sparsity': sparsity,
    'Basket rows': basket_rows,
    'Members': n_members,
    'Distinct baskets': n_baskets,
    'Products': n_products,
    'Unparsed dates': null_dates,
}

display(pd.DataFrame([DATA_KPIS]))
print(f'Rating scale: {rating_min} → {rating_max}; mean={rating_mean:.3f}; sparsity={sparsity:.2%}')

# ---- source cell 11 ----
# Visual health check
rating_dist_pd = ratings_df.groupBy('rating').count().orderBy('rating').toPandas()
fig_rating_dist = px.bar(
    rating_dist_pd, x='rating', y='count',
    title='Rating distribution',
    labels={'rating':'Rating', 'count':'Number of ratings'}
)
fig_rating_dist.update_traces(hovertemplate='Rating=%{x}<br>Count=%{y}<extra></extra>')
fig_rating_dist.show()


## 2. Spark RDD/DataFrame analytics

**Mục tiêu:** thực hiện các yêu cầu Big Data nền tảng bằng cả RDD và DataFrame để minh chứng cách Spark xử lý cùng một bài toán ở hai abstraction level.

**Output:** top products/customers, xu hướng theo thời gian và các bảng/biểu đồ đối chiếu.

**Điểm cần trình bày:** DataFrame thường dễ tối ưu hơn nhờ Catalyst; RDD cho thấy rõ phép biến đổi phân tán.


In [ ]:
# Auto-generated from BIG_DATA_Customer_Intelligence_PRO_Colab.ipynb

# ---- source cell 13 ----
def read_csv_rdd(file_path):
    rdd = sc.textFile(file_path)
    header = rdd.first()
    return rdd.filter(lambda line: line != header).map(lambda line: line.split(','))

def f1_rdd(file_path, top_n=100):
    return (
        read_csv_rdd(file_path)
        .map(lambda x: (x[2], 1))
        .reduceByKey(lambda a, b: a + b)
        .takeOrdered(top_n, key=lambda x: -x[1])
    )

def f2_rdd(file_path, top_n=100):
    # Basket = Member_number + Date.
    return (
        read_csv_rdd(file_path)
        .map(lambda x: ((x[0], x[1]), 1))
        .keys().distinct()
        .map(lambda x: (x[0], 1))
        .reduceByKey(lambda a, b: a + b)
        .takeOrdered(top_n, key=lambda x: -x[1])
    )

def f3_rdd(file_path):
    return (
        read_csv_rdd(file_path)
        .map(lambda x: ((int(x[3]), int(x[4]), x[2]), 1))
        .reduceByKey(lambda a, b: a + b)
    )

def f4_rdd(file_path):
    return (
        read_csv_rdd(file_path)
        .map(lambda x: ((x[0], x[1]), (int(x[3]), int(x[4]))))
        .distinct()
        .map(lambda kv: (kv[1], 1))
        .reduceByKey(lambda a, b: a + b)
        .sortByKey()
        .collect()
    )

# ---- source cell 14 ----
# DataFrame equivalents
f1_df = (
    baskets_df.groupBy('itemDescription').count()
    .orderBy(F.desc('count')).limit(100)
)

f2_df = (
    baskets_df.select('Member_number', 'Date').distinct()
    .groupBy('Member_number').count()
    .orderBy(F.desc('count')).limit(100)
)

f4_df = (
    baskets_df.select('Member_number', 'Date', 'year', 'month').distinct()
    .groupBy('year', 'month').count()
    .orderBy('year', 'month')
)

print('Top products — RDD')
display(pd.DataFrame(f1_rdd(BASKETS_PATH, 20), columns=['product','count']))
print('Top customers by distinct baskets — RDD')
display(pd.DataFrame(f2_rdd(BASKETS_PATH, 20), columns=['member','basket_count']))
print('Monthly distinct baskets — RDD')
display(pd.DataFrame(f4_rdd(BASKETS_PATH), columns=['year_month','basket_count']))

# ---- source cell 15 ----
top_products_pd = f1_df.limit(20).toPandas().sort_values('count')
fig_top_products = px.bar(
    top_products_pd, x='count', y='itemDescription', orientation='h',
    title='Top 20 purchased products',
    labels={'count':'Purchases', 'itemDescription':'Product'}
)
fig_top_products.update_traces(hovertemplate='%{y}<br>Purchases=%{x}<extra></extra>')
fig_top_products.show()


## 3. RFV Customer Analytics

**Mục tiêu:** tạo phân khúc khách hàng kiểu RFM nhưng không bịa dữ liệu tiền tệ. Vì `baskets.csv` không có doanh thu, notebook dùng **RFV = Recency–Frequency–Volume**.

**Output:** RFV score, lifecycle segment và trực quan hóa customer segments.

**Ý nghĩa kinh doanh:** giúp mô tả nhóm khách hàng active, loyal, at-risk và low-engagement bằng hành vi mua thực tế.


In [ ]:
# Auto-generated from BIG_DATA_Customer_Intelligence_PRO_Colab.ipynb

# ---- source cell 17 ----
max_date = baskets_df.agg(F.max('date_parsed')).first()[0]

basket_level = (
    baskets_df.filter(F.col('date_parsed').isNotNull())
    .groupBy('Member_number', 'date_parsed')
    .agg(F.count('*').alias('basket_size'))
)

rfv_df = (
    basket_level.groupBy('Member_number')
    .agg(
        F.datediff(F.lit(max_date), F.max('date_parsed')).alias('recency'),
        F.count('*').alias('frequency'),
        F.sum('basket_size').alias('value_proxy'),
        F.avg('basket_size').alias('avg_basket_size')
    )
)

w_r = Window.orderBy(F.col('recency').asc())
w_f = Window.orderBy(F.col('frequency').asc())
w_v = Window.orderBy(F.col('value_proxy').asc())

rfv_df = (
    rfv_df
    .withColumn('R', 6 - F.ntile(5).over(w_r))
    .withColumn('F', F.ntile(5).over(w_f))
    .withColumn('V', F.ntile(5).over(w_v))
    .withColumn('RFV_score', F.col('R') + F.col('F') + F.col('V'))
    .withColumn(
        'segment',
        F.when((F.col('R') >= 4) & (F.col('F') >= 4), 'Champions')
         .when((F.col('F') >= 4) & (F.col('R') >= 2), 'Loyal')
         .when((F.col('R') >= 4) & (F.col('F') <= 2), 'New / Promising')
         .when((F.col('R') <= 2) & (F.col('F') >= 4), 'At Risk')
         .when((F.col('R') <= 2) & (F.col('F') <= 2), 'Hibernating')
         .otherwise('Regular')
    )
)
rfv_pd = rfv_df.toPandas()
display(rfv_pd.groupby('segment').agg(
    customers=('Member_number','count'),
    recency=('recency','mean'),
    frequency=('frequency','mean'),
    value_proxy=('value_proxy','mean')
).round(2).sort_values('customers', ascending=False))

# ---- source cell 18 ----
segment_summary = (
    rfv_pd.groupby('segment', as_index=False)
    .agg(customers=('Member_number','count'), recency=('recency','mean'),
         frequency=('frequency','mean'), value_proxy=('value_proxy','mean'))
)
fig_rfv = px.treemap(
    segment_summary, path=['segment'], values='customers',
    color='frequency',
    hover_data={'recency':':.1f','frequency':':.1f','value_proxy':':.1f'},
    title='RFV customer portfolio — hover to explain each segment'
)
fig_rfv.show()


## 4. Market Basket Intelligence — FP-Growth

**Mục tiêu:** khai phá Market Basket bằng Spark FP-Growth, tuning support/confidence và ưu tiên rule có tính hành động.

**Output:** frequent itemsets, association rules, Confidence/Lift/Support và cross-sell visualization.

**Ý nghĩa:** Lift giúp phân biệt một rule thật sự có quan hệ với rule chỉ xuất hiện do item vốn đã phổ biến.


In [ ]:
# Auto-generated from BIG_DATA_Customer_Intelligence_PRO_Colab.ipynb

# ---- source cell 20 ----
fp_baskets = (
    baskets_df.groupBy('Member_number', 'Date')
    .agg(F.collect_set('itemDescription').alias('Items'))
    .filter(F.size('Items') >= 2)
    .cache()
)
print('Baskets used by FP-Growth:', fp_baskets.count())

# ---- source cell 21 ----
FP_SUPPORTS = [0.003, 0.005, 0.01, 0.02] if QUALITY_MODE == 'max' else [0.005, 0.01, 0.02]
FP_CONFIDENCES = [0.05, 0.10, 0.20] if QUALITY_MODE == 'max' else [0.05, 0.10]

fp_trials = []
for min_support, min_conf in itertools.product(FP_SUPPORTS, FP_CONFIDENCES):
    t0 = time.perf_counter()
    model = FPGrowth(itemsCol='Items', minSupport=min_support, minConfidence=min_conf).fit(fp_baskets)
    rules = model.associationRules
    s = rules.agg(
        F.count('*').alias('n_rules'),
        F.avg('confidence').alias('avg_confidence'),
        F.avg('lift').alias('avg_lift'),
        F.max('lift').alias('max_lift'),
        F.avg('support').alias('avg_support')
    ).first()
    n_rules = int(s['n_rules'] or 0)
    avg_conf = float(s['avg_confidence'] or 0)
    avg_lift = float(s['avg_lift'] or 0)
    avg_support = float(s['avg_support'] or 0)
    # Prefer useful but reviewable rule volume. Hard penalty for pathological output sizes.
    volume_factor = min(1.0, n_rules / 30.0) * min(1.0, 800.0 / max(n_rules, 1))
    actionability = volume_factor * avg_conf * max(avg_lift, 0) * math.sqrt(max(avg_support, 1e-9))
    fp_trials.append({
        'minSupport': min_support, 'minConfidence': min_conf, 'rules': n_rules,
        'avg_confidence': avg_conf, 'avg_lift': avg_lift, 'max_lift': float(s['max_lift'] or 0),
        'avg_support': avg_support, 'actionability': actionability,
        'runtime_sec': time.perf_counter() - t0
    })

fp_tuning_pd = pd.DataFrame(fp_trials).sort_values('actionability', ascending=False)
display(fp_tuning_pd)
best_fp = fp_tuning_pd.iloc[0].to_dict()
print('Selected FP-Growth params:', best_fp)

# ---- source cell 22 ----
fp_model = FPGrowth(
    itemsCol='Items',
    minSupport=float(best_fp['minSupport']),
    minConfidence=float(best_fp['minConfidence'])
).fit(fp_baskets)

fp_rules = (
    fp_model.associationRules
    .withColumn('action_score', F.col('confidence') * F.col('lift') * F.sqrt(F.col('support')))
    .orderBy(F.desc('action_score'))
)
fp_rules_pd = fp_rules.limit(200).toPandas()
fp_rules_pd['antecedent_text'] = fp_rules_pd['antecedent'].apply(lambda x: ' + '.join(x))
fp_rules_pd['consequent_text'] = fp_rules_pd['consequent'].apply(lambda x: ' + '.join(x))
display(fp_rules_pd[['antecedent_text','consequent_text','confidence','lift','support','action_score']].head(30))

# ---- source cell 23 ----
fig_rules = px.scatter(
    fp_rules_pd,
    x='confidence', y='lift', size='support', color='action_score',
    hover_name='antecedent_text', hover_data={'consequent_text':True,'support':':.4f','action_score':':.4f'},
    title='Association Rules — Confidence × Lift × Support'
)
fig_rules.add_hline(y=1.0, line_dash='dash', annotation_text='Lift = 1 (no positive association)')
fig_rules.show()

# ---- source cell 24 ----
# Interactive product-rule network for top actionable rules.
def build_rule_network(rules_pd, n=25):
    d = rules_pd.head(n).copy()
    G = nx.DiGraph()
    for _, r in d.iterrows():
        a = r['antecedent_text']; c = r['consequent_text']
        G.add_edge(a, c, confidence=float(r['confidence']), lift=float(r['lift']), support=float(r['support']))
    pos = nx.spring_layout(G, seed=SEED, k=1.3)
    edge_x, edge_y = [], []
    mid_x, mid_y, mid_text = [], [], []
    for u, v, data in G.edges(data=True):
        x0,y0 = pos[u]; x1,y1 = pos[v]
        edge_x += [x0,x1,None]; edge_y += [y0,y1,None]
        mid_x.append((x0+x1)/2); mid_y.append((y0+y1)/2)
        mid_text.append(f'{u} → {v}<br>confidence={data["confidence"]:.3f}<br>lift={data["lift"]:.3f}<br>support={data["support"]:.4f}')
    edge_trace = go.Scatter(x=edge_x,y=edge_y,mode='lines',hoverinfo='skip',line=dict(width=1,color='#94A3B8'))
    mid_trace = go.Scatter(x=mid_x,y=mid_y,mode='markers',marker=dict(size=10,opacity=0.01),text=mid_text,hovertemplate='%{text}<extra></extra>')
    node_x,node_y,node_text,node_degree=[],[],[],[]
    for node in G.nodes():
        x,y=pos[node]; node_x.append(x); node_y.append(y); node_text.append(node); node_degree.append(G.degree(node))
    node_trace = go.Scatter(
        x=node_x,y=node_y,mode='markers+text',text=node_text,textposition='top center',
        marker=dict(size=[12+3*d for d in node_degree],color=node_degree,colorscale='Blues',showscale=True,colorbar=dict(title='Degree')),
        hovertemplate='%{text}<extra></extra>'
    )
    fig=go.Figure([edge_trace,mid_trace,node_trace])
    fig.update_layout(title='Cross-sell network — hover edges for rule evidence',showlegend=False,height=650,
                      xaxis=dict(visible=False),yaxis=dict(visible=False),margin=dict(l=10,r=10,t=55,b=10))
    return fig

fig_rule_network = build_rule_network(fp_rules_pd, 25)
fig_rule_network.show()


## 5. Distributed SVD & latent representation

**Mục tiêu:** giảm chiều ma trận user–item bằng truncated SVD để học latent representation có thể dùng cho clustering và recommendation.

**Output:** singular-value energy curve, số chiều được chọn có căn cứ, user/item embeddings và latent concepts.

**Lý do chọn số chiều:** dựa trên captured energy thay vì chọn một con số tùy ý.


In [ ]:
# Auto-generated from BIG_DATA_Customer_Intelligence_PRO_Colab.ipynb

# ---- source cell 26 ----
user_means_spark = ratings_df.groupBy('user').agg(F.avg('rating').alias('user_mean'))
centered_ratings = (
    ratings_df.join(user_means_spark, 'user')
    .withColumn('centered_rating', F.col('rating') - F.col('user_mean'))
)

item_ids = [r['item'] for r in ratings_df.select('item').distinct().orderBy('item').collect()]
user_item_centered = (
    centered_ratings.groupBy('user').pivot('item', item_ids)
    .agg(F.first('centered_rating')).na.fill(0.0).orderBy('user')
)
user_ids = [int(r['user']) for r in user_item_centered.select('user').collect()]
feature_cols = [str(i) for i in item_ids]

rows_rdd = user_item_centered.select(feature_cols).rdd.map(
    lambda r: OldVectors.dense([float(x or 0.0) for x in r])
).cache()
row_matrix = RowMatrix(rows_rdd)

max_possible = max(2, min(len(user_ids)-1, len(item_ids)-1, 64))
SVD_CANDIDATES = [k for k in [4,8,16,32,64] if k <= max_possible]
max_k = max(SVD_CANDIDATES)
print('SVD candidates:', SVD_CANDIDATES, '| max_k:', max_k)

# ---- source cell 27 ----
t0 = time.perf_counter()
svd_full = row_matrix.computeSVD(max_k, computeU=True)
svd_runtime = time.perf_counter() - t0

singular_values = np.array(svd_full.s, dtype=float)
total_energy = rows_rdd.map(lambda v: float(np.dot(v.toArray(), v.toArray()))).sum()
cum_energy = np.cumsum(singular_values**2) / max(total_energy, 1e-12)

svd_curve = pd.DataFrame({
    'component': np.arange(1, len(singular_values)+1),
    'singular_value': singular_values,
    'captured_energy': cum_energy
})

selected_svd_k = None
for k in SVD_CANDIDATES:
    if cum_energy[k-1] >= 0.90:
        selected_svd_k = k
        break
if selected_svd_k is None:
    selected_svd_k = max_k

print(f'Selected latent dimension = {selected_svd_k}; captured energy={cum_energy[selected_svd_k-1]:.2%}; runtime={svd_runtime:.2f}s')

# ---- source cell 28 ----
fig_svd = make_subplots(specs=[[{'secondary_y': True}]])
fig_svd.add_trace(go.Bar(x=svd_curve['component'], y=svd_curve['singular_value'], name='Singular value'), secondary_y=False)
fig_svd.add_trace(go.Scatter(x=svd_curve['component'], y=svd_curve['captured_energy'], name='Cumulative captured energy', mode='lines+markers'), secondary_y=True)
fig_svd.add_hline(y=0.90, line_dash='dash', annotation_text='90% target', secondary_y=True)
fig_svd.add_vline(x=selected_svd_k, line_dash='dot', annotation_text=f'Selected k={selected_svd_k}')
fig_svd.update_layout(title='SVD latent-dimension evidence', hovermode='x unified')
fig_svd.update_yaxes(title_text='Singular value', secondary_y=False)
fig_svd.update_yaxes(title_text='Captured energy', tickformat='.0%', secondary_y=True)
fig_svd.show()

# ---- source cell 29 ----
# Extract UΣ and VΣ embeddings.
U_np = np.vstack([v.toArray() for v in svd_full.U.rows.collect()])[:, :selected_svd_k]
S_np = singular_values[:selected_svd_k]
V_np = svd_full.V.toArray()[:, :selected_svd_k]

user_embeddings = U_np * S_np
item_embeddings = V_np * S_np

user_emb_pd = pd.DataFrame(user_embeddings, columns=[f'z{i+1}' for i in range(selected_svd_k)])
user_emb_pd.insert(0, 'user', user_ids)
item_emb_pd = pd.DataFrame(item_embeddings, columns=[f'z{i+1}' for i in range(selected_svd_k)])
item_emb_pd.insert(0, 'item', item_ids)

user_emb_pd.to_csv(OUTPUT_DIR/'user_svd_embeddings.csv', index=False)
item_emb_pd.to_csv(OUTPUT_DIR/'item_svd_embeddings.csv', index=False)
display(user_emb_pd.head())

# ---- source cell 30 ----
# Preserve the original Task 1 deliverables: assign each user/item to its dominant latent concept.
# Dominant concept = latent dimension with the largest absolute embedding magnitude.
user_concepts_pd = pd.DataFrame({
    'user': user_ids,
    'concept': np.argmax(np.abs(user_embeddings), axis=1).astype(int) + 1,
    'concept_strength': np.max(np.abs(user_embeddings), axis=1)
})
item_concepts_pd = pd.DataFrame({
    'item': item_ids,
    'concept': np.argmax(np.abs(item_embeddings), axis=1).astype(int) + 1,
    'concept_strength': np.max(np.abs(item_embeddings), axis=1)
})
user_concepts_pd.to_csv(OUTPUT_DIR/'concept_user.csv', index=False)
item_concepts_pd.to_csv(OUTPUT_DIR/'concept_item.csv', index=False)
display(user_concepts_pd.head())
display(item_concepts_pd.head())


## 6. CURE — real representative-point clustering

**Mục tiêu:** chạy **CURE đúng bản chất** với nhiều representative points và compression, không dùng KMeans rồi đổi tên.

**Output:** tuning theo số cluster/representatives/compression, cluster labels và representative points.

**Đánh giá:** Silhouette, Davies–Bouldin, Calinski–Harabasz và độ cân bằng cluster.


In [ ]:
# Auto-generated from BIG_DATA_Customer_Intelligence_PRO_Colab.ipynb

# ---- source cell 32 ----
scaled_embeddings = StandardScaler().fit_transform(user_embeddings)
vis_n = min(len(user_ids), 3000)
vis_idx = np.random.default_rng(SEED).choice(len(user_ids), vis_n, replace=False)
X_vis = scaled_embeddings[vis_idx]
users_vis = np.array(user_ids)[vis_idx]

pca_2d = PCA(n_components=2, random_state=SEED).fit_transform(X_vis)
umap_2d = umap.UMAP(n_components=2, n_neighbors=min(15, max(5, vis_n//20)), min_dist=0.1, random_state=SEED).fit_transform(X_vis)
perplexity = min(30, max(5, (vis_n-1)//3))
tsne_2d = TSNE(n_components=2, perplexity=perplexity, init='pca', learning_rate='auto', random_state=SEED).fit_transform(X_vis)

vis_pd = pd.DataFrame({
    'user': users_vis,
    'PCA-1': pca_2d[:,0], 'PCA-2': pca_2d[:,1],
    'UMAP-1': umap_2d[:,0], 'UMAP-2': umap_2d[:,1],
    'tSNE-1': tsne_2d[:,0], 'tSNE-2': tsne_2d[:,1],
})
for method, x, y in [('PCA','PCA-1','PCA-2'), ('UMAP','UMAP-1','UMAP-2'), ('t-SNE','tSNE-1','tSNE-2')]:
    fig = px.scatter(vis_pd, x=x, y=y, hover_data=['user'], title=f'{method} projection of SVD user embeddings')
    fig.update_traces(marker=dict(size=6, opacity=.65))
    fig.show()

# ---- source cell 34 ----
from pyclustering.cluster.cure import cure

CURE_MAX_FULL = 5000
CURE_TUNE_SAMPLE = min(len(user_ids), 1200 if QUALITY_MODE == 'max' else 700)
CURE_KS = list(range(3, min(9, max(4, len(user_ids)//10))))
CURE_REPS = [5, 10] if QUALITY_MODE == 'max' else [5]
CURE_COMPRESSIONS = [0.3, 0.5] if QUALITY_MODE == 'max' else [0.5]

rng = np.random.default_rng(SEED)
tune_idx = rng.choice(len(scaled_embeddings), CURE_TUNE_SAMPLE, replace=False)
X_cure_tune = scaled_embeddings[tune_idx]

def fit_cure_labels(X, k, reps=5, compression=0.5):
    # CCORE first; fallback to Python implementation for compatibility.
    try:
        inst = cure(X.tolist(), k, reps, compression, ccore=True)
        inst.process()
    except Exception:
        inst = cure(X.tolist(), k, reps, compression, ccore=False)
        inst.process()
    clusters = inst.get_clusters()
    labels = np.full(len(X), -1, dtype=int)
    for cid, ids in enumerate(clusters):
        labels[np.array(ids, dtype=int)] = cid
    reps_out = [[np.asarray(p, dtype=float) for p in cluster_reps] for cluster_reps in inst.get_representors()]
    return labels, reps_out

def safe_cluster_metrics(X, labels):
    unique, counts = np.unique(labels, return_counts=True)
    if len(unique) < 2 or np.min(counts) < 2:
        return np.nan, np.nan, np.nan, 0.0
    sil = silhouette_score(X, labels)
    db = davies_bouldin_score(X, labels)
    ch = calinski_harabasz_score(X, labels)
    balance = float(np.min(counts) / np.max(counts))
    return sil, db, ch, balance

# ---- source cell 35 ----
cure_trials=[]
for k, reps, comp in itertools.product(CURE_KS, CURE_REPS, CURE_COMPRESSIONS):
    t0=time.perf_counter()
    labels, _ = fit_cure_labels(X_cure_tune, k, reps, comp)
    sil, db, ch, balance = safe_cluster_metrics(X_cure_tune, labels)
    cure_trials.append({
        'k':k, 'representatives':reps, 'compression':comp,
        'silhouette':sil, 'davies_bouldin':db, 'calinski_harabasz':ch,
        'balance':balance, 'runtime_sec':time.perf_counter()-t0
    })

cure_tuning_pd=pd.DataFrame(cure_trials).dropna().copy()
if cure_tuning_pd.empty:
    raise RuntimeError('CURE tuning produced no valid multi-cluster solution. Reduce k/representatives or inspect the embedding.')
# Rank-based composite avoids arbitrary metric scales.
cure_tuning_pd['r_sil'] = cure_tuning_pd['silhouette'].rank(pct=True)
cure_tuning_pd['r_db'] = (-cure_tuning_pd['davies_bouldin']).rank(pct=True)
cure_tuning_pd['r_ch'] = cure_tuning_pd['calinski_harabasz'].rank(pct=True)
cure_tuning_pd['cluster_score'] = (
    .40*cure_tuning_pd['r_sil'] + .20*cure_tuning_pd['r_db'] +
    .20*cure_tuning_pd['r_ch'] + .20*cure_tuning_pd['balance']
)
cure_tuning_pd=cure_tuning_pd.sort_values('cluster_score', ascending=False)
display(cure_tuning_pd.head(15))
best_cure=cure_tuning_pd.iloc[0].to_dict()
print('Selected CURE:', best_cure)

# ---- source cell 36 ----
def assign_to_representatives(X, cluster_reps):
    out=np.empty(len(X), dtype=int)
    for i, x in enumerate(X):
        distances=[]
        for reps in cluster_reps:
            R=np.vstack(reps)
            distances.append(float(np.min(np.linalg.norm(R-x, axis=1))))
        out[i]=int(np.argmin(distances))
    return out

def spark_assign_to_representatives(user_ids_local, X, cluster_reps):
    """Distributed nearest-representative assignment for the sampled-CURE branch."""
    reps_serializable=[[p.astype(float).tolist() for p in reps] for reps in cluster_reps]
    bc=sc.broadcast(reps_serializable)
    rows=[(int(u), [float(v) for v in x]) for u,x in zip(user_ids_local,X)]
    sdf=spark.createDataFrame(rows, schema=T.StructType([
        T.StructField('user',T.IntegerType(),False),
        T.StructField('embedding',T.ArrayType(T.DoubleType()),False)
    ]))
    @F.udf(T.IntegerType())
    def nearest_cluster(arr):
        x=np.asarray(arr,dtype=float); ds=[]
        for reps in bc.value:
            R=np.asarray(reps,dtype=float)
            ds.append(float(np.min(np.linalg.norm(R-x,axis=1))))
        return int(np.argmin(ds))
    assigned=sdf.withColumn('cluster',nearest_cluster('embedding')).select('user','cluster')
    mapping={int(r['user']):int(r['cluster']) for r in assigned.collect()}
    bc.unpersist()
    return np.array([mapping[int(u)] for u in user_ids_local],dtype=int)

# Fit final CURE.
if len(scaled_embeddings) <= CURE_MAX_FULL:
    cure_labels, cure_reps = fit_cure_labels(
        scaled_embeddings, int(best_cure['k']), int(best_cure['representatives']), float(best_cure['compression'])
    )
    cure_mode='full CURE'
else:
    sample_idx=np.random.default_rng(SEED).choice(len(scaled_embeddings), CURE_MAX_FULL, replace=False)
    _, cure_reps=fit_cure_labels(
        scaled_embeddings[sample_idx], int(best_cure['k']), int(best_cure['representatives']), float(best_cure['compression'])
    )
    cure_labels=spark_assign_to_representatives(user_ids, scaled_embeddings, cure_reps)
    cure_mode='sampled CURE + Spark representative assignment'

cure_sil, cure_db, cure_ch, cure_balance = safe_cluster_metrics(scaled_embeddings, cure_labels)
print(cure_mode, '| silhouette=', round(cure_sil,4), '| DB=', round(cure_db,4), '| CH=', round(cure_ch,2))


### 6.1 Stability, personas & visual story

**Mục tiêu:** kiểm tra độ ổn định của CURE và biến cluster thành câu chuyện có thể giải thích cho khách hàng.

**Output:** stability ARI, PCA/UMAP/t-SNE, cluster personas và profile visualization.

**Ý nghĩa:** cluster tốt không chỉ tách nhau trên metric mà còn phải ổn định và diễn giải được.


In [ ]:
# Auto-generated from BIG_DATA_Customer_Intelligence_PRO_Colab.ipynb

# ---- source cell 37 ----
# Stability validation: fit on repeated 80% samples, then assign all users by CURE representatives.
stability_labelings=[]
stability_sil=[]
for seed in STAT_SEEDS:
    idx=np.random.default_rng(seed).choice(len(scaled_embeddings), max(50, int(.80*len(scaled_embeddings))), replace=False)
    _, reps=fit_cure_labels(
        scaled_embeddings[idx], int(best_cure['k']), int(best_cure['representatives']), float(best_cure['compression'])
    )
    full_labels=assign_to_representatives(scaled_embeddings, reps)
    stability_labelings.append(full_labels)
    stability_sil.append(silhouette_score(scaled_embeddings, full_labels))

ari_values=[]
for a,b in itertools.combinations(stability_labelings,2):
    ari_values.append(adjusted_rand_score(a,b))
cure_stability_ari=float(np.mean(ari_values)) if ari_values else 1.0
print('CURE stability ARI mean=', round(cure_stability_ari,4), '| silhouette mean±std=', round(np.mean(stability_sil),4), '±', round(np.std(stability_sil),4))

# ---- source cell 39 ----
user_behavior_pd = (
    ratings_df.groupBy('user')
    .agg(
        F.count('*').alias('rating_count'),
        F.avg('rating').alias('avg_rating'),
        F.stddev('rating').alias('rating_std'),
        F.countDistinct('item').alias('item_diversity')
    ).toPandas()
)
cluster_users_pd = user_emb_pd[['user']].copy()
cluster_users_pd['cluster'] = cure_labels
cluster_users_pd = cluster_users_pd.merge(user_behavior_pd, on='user', how='left')

cluster_profile = (
    cluster_users_pd.groupby('cluster', as_index=False)
    .agg(users=('user','count'), rating_count=('rating_count','mean'), avg_rating=('avg_rating','mean'),
         rating_std=('rating_std','mean'), item_diversity=('item_diversity','mean'))
)

activity_median=cluster_profile['rating_count'].median()
rating_median=cluster_profile['avg_rating'].median()
def persona(row):
    activity='High-activity' if row.rating_count >= activity_median else 'Selective'
    sentiment='high-rating' if row.avg_rating >= rating_median else 'critical-rating'
    return f'{activity} / {sentiment}'
cluster_profile['persona']=cluster_profile.apply(persona, axis=1)
cluster_users_pd=cluster_users_pd.merge(cluster_profile[['cluster','persona']],on='cluster',how='left')
display(cluster_profile.round(3))

# ---- source cell 40 ----
# Reuse UMAP coordinates for visual cluster story.
vis_cluster = vis_pd.copy()
label_map=dict(zip(user_ids, cure_labels))
profile_map=dict(zip(cluster_profile['cluster'], cluster_profile['persona']))
vis_cluster['cluster']=vis_cluster['user'].map(label_map).astype(str)
vis_cluster['persona']=vis_cluster['user'].map(lambda u: profile_map[label_map[u]])
vis_cluster=vis_cluster.merge(user_behavior_pd,on='user',how='left')

fig_cluster_umap=px.scatter(
    vis_cluster, x='UMAP-1', y='UMAP-2', color='cluster',
    hover_data=['user','persona','rating_count','avg_rating','item_diversity'],
    title='CURE segmentation on SVD embeddings — interactive UMAP'
)
fig_cluster_umap.update_traces(marker=dict(size=7, opacity=.72))
fig_cluster_umap.show()

# ---- source cell 41 ----
# Radar-style normalized cluster profile.
profile_features=['rating_count','avg_rating','rating_std','item_diversity']
prof=cluster_profile.copy()
for c in profile_features:
    lo,hi=prof[c].min(),prof[c].max()
    prof[c+'_n']=(prof[c]-lo)/(hi-lo+1e-9)
fig_cluster_radar=go.Figure()
for _,r in prof.iterrows():
    vals=[r[c+'_n'] for c in profile_features]
    fig_cluster_radar.add_trace(go.Scatterpolar(
        r=vals+[vals[0]], theta=profile_features+[profile_features[0]], fill='toself',
        name=f"Cluster {int(r['cluster'])}: {r['persona']}"
    ))
fig_cluster_radar.update_layout(title='Cluster persona profile (normalized)', polar=dict(radialaxis=dict(visible=True,range=[0,1])))
fig_cluster_radar.show()


## 7. Recommender benchmarks — baselines, UserCF, SVD

**Mục tiêu:** xây baseline và nhiều recommender cạnh tranh công bằng: global/user/item mean, UserCF và SVD-based reconstruction.

**Đánh giá hai mục tiêu:** rating prediction bằng RMSE/MAE và Top-N relevance bằng Precision@K/Recall@K.

**Nguyên tắc:** không dùng test ratings trong lúc fit để tránh data leakage.


In [ ]:
# Auto-generated from BIG_DATA_Customer_Intelligence_PRO_Colab.ipynb

# ---- source cell 43 ----
ratings_pd = ratings_df.toPandas().astype({'user':int,'item':int,'rating':float})
RATING_LOW=float(ratings_pd['rating'].min()); RATING_HIGH=float(ratings_pd['rating'].max())
if RATING_HIGH < RELEVANCE_THRESHOLD:
    RELEVANCE_THRESHOLD=float(ratings_pd['rating'].quantile(.75))
print('Relevance threshold =', RELEVANCE_THRESHOLD)

def per_user_split(df, seed=42, test_frac=.2, min_ratings=5):
    rng=np.random.default_rng(seed)
    train_idx=[]; test_idx=[]
    for _,g in df.groupby('user'):
        ids=g.index.to_numpy()
        if len(ids) < min_ratings:
            train_idx.extend(ids.tolist()); continue
        n_test=max(1,int(round(len(ids)*test_frac)))
        n_test=min(n_test, len(ids)-2)
        chosen=set(rng.choice(ids, n_test, replace=False).tolist())
        test_idx.extend(chosen)
        train_idx.extend([i for i in ids if i not in chosen])
    train=df.loc[train_idx].copy(); test=df.loc[test_idx].copy()
    # prevent item cold start in test
    train_items=set(train['item'])
    cold=test[~test['item'].isin(train_items)]
    if len(cold):
        train=pd.concat([train,cold],ignore_index=False)
        test=test.drop(index=cold.index)
    return train.reset_index(drop=True), test.reset_index(drop=True)

def rating_metrics(y_true,y_pred):
    return {
        'RMSE': float(np.sqrt(mean_squared_error(y_true,y_pred))),
        'MAE': float(mean_absolute_error(y_true,y_pred))
    }

def ranking_metrics(pred_dict, test_df, k=10, threshold=4.0):
    relevant=(test_df[test_df['rating']>=threshold].groupby('user')['item'].apply(set).to_dict())
    users=[u for u,s in relevant.items() if len(s)>0 and u in pred_dict]
    if not users: return {'Precision@K':0.0,'Recall@K':0.0,'eval_users':0}
    ps=[]; rs=[]
    for u in users:
        pred=list(pred_dict[u])[:k]; gt=relevant[u]
        hits=len(set(pred)&gt)
        ps.append(hits/k)
        rs.append(hits/len(gt))
    return {'Precision@K':float(np.mean(ps)),'Recall@K':float(np.mean(rs)),'eval_users':len(users)}

# ---- source cell 44 ----
# ---------- UserCF ----------
def build_usercf_state(train):
    users=sorted(train.user.unique()); items=sorted(train.item.unique())
    pivot=train.pivot_table(index='user',columns='item',values='rating',aggfunc='mean').reindex(index=users,columns=items)
    A=pivot.to_numpy(dtype=float); mask=~np.isnan(A)
    global_mean=float(train.rating.mean())
    user_mean=np.nanmean(A,axis=1)
    user_mean=np.where(np.isnan(user_mean),global_mean,user_mean)
    centered=np.where(mask,A-user_mean[:,None],0.0)
    norms=np.linalg.norm(centered,axis=1)
    sim=centered@centered.T/(np.outer(norms,norms)+1e-12)
    sim=np.where(sim>0,sim,0.0); np.fill_diagonal(sim,0.0)
    return {'users':users,'items':items,'u2i':{u:i for i,u in enumerate(users)},'i2i':{it:i for i,it in enumerate(items)},
            'A':A,'mask':mask,'user_mean':user_mean,'centered':centered,'sim':sim,'global_mean':global_mean,
            'item_mean':train.groupby('item').rating.mean().to_dict()}

def usercf_similarity_topn(sim,n_neighbors):
    if n_neighbors>=sim.shape[1]: return sim.copy()
    out=np.zeros_like(sim)
    idx=np.argpartition(sim,-n_neighbors,axis=1)[:,-n_neighbors:]
    rows=np.arange(sim.shape[0])[:,None]
    out[rows,idx]=sim[rows,idx]
    return out

def usercf_predict(state, pairs, n_neighbors=40):
    sim=usercf_similarity_topn(state['sim'],n_neighbors)
    pred=[]
    for r in pairs.itertuples():
        if r.user not in state['u2i']:
            p=state['item_mean'].get(r.item,state['global_mean']); pred.append(p); continue
        ui=state['u2i'][r.user]
        if r.item not in state['i2i']:
            pred.append(state['user_mean'][ui]); continue
        ii=state['i2i'][r.item]
        raters=state['mask'][:,ii]
        w=sim[ui]*raters
        den=np.sum(np.abs(w))
        p=state['user_mean'][ui] if den<1e-12 else state['user_mean'][ui]+np.sum(w*state['centered'][:,ii])/den
        pred.append(float(np.clip(p,RATING_LOW,RATING_HIGH)))
    return np.asarray(pred)

def usercf_recommend(state, users, k=10, n_neighbors=40):
    sim=usercf_similarity_topn(state['sim'],n_neighbors)
    num=sim@state['centered']; den=np.abs(sim)@state['mask'].astype(float)
    score=state['user_mean'][:,None]+np.divide(num,den,out=np.zeros_like(num),where=den>1e-12)
    out={}
    items=np.array(state['items'])
    for u in users:
        if u not in state['u2i']: continue
        ui=state['u2i'][u]; s=score[ui].copy(); s[state['mask'][ui]]=-np.inf
        take=min(k,len(s)); idx=np.argpartition(s,-take)[-take:]; idx=idx[np.argsort(s[idx])[::-1]]
        out[u]=items[idx].astype(int).tolist()
    return out

# ---- source cell 45 ----
# ---------- SVD recommender ----------
def build_svd_recommender(train, n_components=16):
    users=sorted(train.user.unique()); items=sorted(train.item.unique())
    pivot=train.pivot_table(index='user',columns='item',values='rating',aggfunc='mean').reindex(index=users,columns=items)
    A=pivot.to_numpy(dtype=float); mask=~np.isnan(A); global_mean=float(train.rating.mean())
    user_mean=np.nanmean(A,axis=1); user_mean=np.where(np.isnan(user_mean),global_mean,user_mean)
    centered=np.where(mask,A-user_mean[:,None],0.0)
    k=max(2,min(n_components,min(centered.shape)-1))
    model=TruncatedSVD(n_components=k,random_state=SEED)
    latent=model.fit_transform(centered); recon=latent@model.components_ + user_mean[:,None]
    return {'users':users,'items':items,'u2i':{u:i for i,u in enumerate(users)},'i2i':{it:i for i,it in enumerate(items)},
            'mask':mask,'pred':np.clip(recon,RATING_LOW,RATING_HIGH),'global_mean':global_mean,
            'user_mean':user_mean,'item_mean':train.groupby('item').rating.mean().to_dict(),'k':k}

def svd_predict(state,pairs):
    out=[]
    for r in pairs.itertuples():
        if r.user in state['u2i'] and r.item in state['i2i']:
            out.append(state['pred'][state['u2i'][r.user],state['i2i'][r.item]])
        elif r.item in state['item_mean']: out.append(state['item_mean'][r.item])
        else: out.append(state['global_mean'])
    return np.asarray(out)

def svd_recommend(state,users,k=10):
    items=np.array(state['items']); out={}
    for u in users:
        if u not in state['u2i']: continue
        ui=state['u2i'][u]; s=state['pred'][ui].copy(); s[state['mask'][ui]]=-np.inf
        take=min(k,len(s)); idx=np.argpartition(s,-take)[-take:]; idx=idx[np.argsort(s[idx])[::-1]]
        out[u]=items[idx].astype(int).tolist()
    return out


### 7.1 Spark ALS + hyperparameter tuning

**Mục tiêu:** tuning Spark ALS theo rank, regularization và iteration để tìm cấu hình tốt trên validation data.

**Output:** bảng hyperparameter search, best ALS configuration và predictions.

**Ý nghĩa Big Data:** ALS là model factorization được Spark MLlib triển khai cho recommendation phân tán.


In [ ]:
# Auto-generated from BIG_DATA_Customer_Intelligence_PRO_Colab.ipynb

# ---- source cell 46 ----
# ---------- Spark ALS ----------
def to_spark_ratings(pdf):
    return spark.createDataFrame(pdf[['user','item','rating']].astype({'user':int,'item':int,'rating':float}))

def fit_als(train_pdf, params, seed=42):
    als=ALS(
        userCol='user', itemCol='item', ratingCol='rating',
        rank=int(params['rank']), regParam=float(params['regParam']), maxIter=int(params['maxIter']),
        coldStartStrategy='drop', seed=int(seed), nonnegative=False
    )
    return als.fit(to_spark_ratings(train_pdf))

def als_rating_eval(model,test_pdf):
    pred=model.transform(to_spark_ratings(test_pdf)).select('rating','prediction').dropna().toPandas()
    if len(pred)==0: return {'RMSE':np.nan,'MAE':np.nan}
    return rating_metrics(pred.rating,pred.prediction)

def als_recommend_dict(model,train_pdf,eval_users,k=10):
    users_sdf=spark.createDataFrame([(int(u),) for u in sorted(set(eval_users))],['user'])
    request_n=min(len(set(train_pdf['item'])),max(100,k*10))
    rec=model.recommendForUserSubset(users_sdf,request_n)
    exploded=(
        rec.select('user',F.posexplode('recommendations').alias('pos','rec'))
        .select('user',F.col('rec.item').alias('item'),F.col('rec.rating').alias('score'))
    )
    seen=to_spark_ratings(train_pdf).select('user','item').distinct()
    unseen=exploded.join(seen,['user','item'],'left_anti')
    w=Window.partitionBy('user').orderBy(F.desc('score'))
    top=(unseen.withColumn('rn',F.row_number().over(w)).filter(F.col('rn')<=k)
         .groupBy('user').agg(F.sort_array(F.collect_list(F.struct('rn','item'))).alias('arr')))
    rows=top.collect()
    return {int(r['user']):[int(x['item']) for x in r['arr']] for r in rows}

# ---- source cell 48 ----
master_train, master_test = per_user_split(ratings_pd, SEED, TEST_FRAC, MIN_USER_RATINGS_FOR_TEST)
inner_train, inner_val = per_user_split(master_train, SEED+100, .15, MIN_USER_RATINGS_FOR_TEST)
val_users=inner_val[inner_val.rating>=RELEVANCE_THRESHOLD].user.unique().tolist()

# UserCF tuning
ucf_state=build_usercf_state(inner_train)
ucf_trials=[]
for nn in ([10,20,40,80] if QUALITY_MODE=='max' else [20,40]):
    p=usercf_predict(ucf_state,inner_val,nn); m=rating_metrics(inner_val.rating,p)
    rec=usercf_recommend(ucf_state,val_users,TOP_K,nn); r=ranking_metrics(rec,inner_val,TOP_K,RELEVANCE_THRESHOLD)
    ucf_trials.append({'neighbors':nn,**m,**r})
ucf_tuning=pd.DataFrame(ucf_trials)
# normalized balanced score
for col in ['RMSE','Recall@K']:
    lo,hi=ucf_tuning[col].min(),ucf_tuning[col].max(); ucf_tuning[col+'_n']=(ucf_tuning[col]-lo)/(hi-lo+1e-9)
ucf_tuning['score']=.6*(1-ucf_tuning['RMSE_n'])+.4*ucf_tuning['Recall@K_n']
best_ucf=int(ucf_tuning.sort_values('score',ascending=False).iloc[0]['neighbors'])
display(ucf_tuning)
print('Best UserCF neighbors=',best_ucf)

# ---- source cell 49 ----
# SVD recommender tuning
svd_rec_candidates=sorted(set([k for k in [4,8,16,32,selected_svd_k] if k < min(inner_train.user.nunique(),inner_train.item.nunique())]))
svd_trials=[]
for k in svd_rec_candidates:
    st=build_svd_recommender(inner_train,k); p=svd_predict(st,inner_val); m=rating_metrics(inner_val.rating,p)
    rec=svd_recommend(st,val_users,TOP_K); r=ranking_metrics(rec,inner_val,TOP_K,RELEVANCE_THRESHOLD)
    svd_trials.append({'components':k,**m,**r})
svd_tuning=pd.DataFrame(svd_trials)
for col in ['RMSE','Recall@K']:
    lo,hi=svd_tuning[col].min(),svd_tuning[col].max(); svd_tuning[col+'_n']=(svd_tuning[col]-lo)/(hi-lo+1e-9)
svd_tuning['score']=.6*(1-svd_tuning['RMSE_n'])+.4*svd_tuning['Recall@K_n']
best_svd_rec_k=int(svd_tuning.sort_values('score',ascending=False).iloc[0]['components'])
display(svd_tuning)
print('Best SVD recommender components=',best_svd_rec_k)

# ---- source cell 50 ----
# ALS tuning — broad enough for demo, bounded to stay below the requested runtime budget.
ALS_RANKS=[8,16,32] if QUALITY_MODE=='max' else [8,16]
ALS_REGS=[0.03,0.10,0.30] if QUALITY_MODE=='max' else [0.05,0.15]
ALS_ITERS=[10,20] if QUALITY_MODE=='max' else [10]

als_trials=[]
for rank,reg,iters in itertools.product(ALS_RANKS,ALS_REGS,ALS_ITERS):
    t0=time.perf_counter(); params={'rank':rank,'regParam':reg,'maxIter':iters}
    model=fit_als(inner_train,params,SEED)
    m=als_rating_eval(model,inner_val)
    als_trials.append({**params,**m,'runtime_sec':time.perf_counter()-t0})
als_tuning=pd.DataFrame(als_trials).sort_values('RMSE')

# Rank quality only on the best RMSE candidates.
top_candidates=als_tuning.head(min(4,len(als_tuning))).copy()
rank_rows=[]
for _,r in top_candidates.iterrows():
    params={'rank':int(r['rank']),'regParam':float(r['regParam']),'maxIter':int(r['maxIter'])}
    model=fit_als(inner_train,params,SEED)
    rec=als_recommend_dict(model,inner_train,val_users,TOP_K)
    rank_rows.append({**params,**ranking_metrics(rec,inner_val,TOP_K,RELEVANCE_THRESHOLD)})
als_rank_tuning=pd.DataFrame(rank_rows)
als_joint=top_candidates.merge(als_rank_tuning,on=['rank','regParam','maxIter'])

lo,hi=als_joint.RMSE.min(),als_joint.RMSE.max(); als_joint['rmse_n']=(als_joint.RMSE-lo)/(hi-lo+1e-9)
lo,hi=als_joint['Recall@K'].min(),als_joint['Recall@K'].max(); als_joint['recall_n']=(als_joint['Recall@K']-lo)/(hi-lo+1e-9)
als_joint['selection_score']=.60*(1-als_joint['rmse_n'])+.40*als_joint['recall_n']
best_als_row=als_joint.sort_values('selection_score',ascending=False).iloc[0]
best_als={'rank':int(best_als_row['rank']),'regParam':float(best_als_row['regParam']),'maxIter':int(best_als_row['maxIter'])}
display(als_joint.sort_values('selection_score',ascending=False))
print('Best ALS params=',best_als)


### 7.2 Statistical validation, model champion & demo

**Mục tiêu:** statistical validation qua nhiều random seeds, tổng hợp RMSE/MAE/Precision@K/Recall@K và chọn **model champion** thay vì dựa vào một split may mắn.

**Output:** mean ± std metrics, champion score và demo Top-N unseen items cho một user.

**Cách đọc:** RMSE/MAE càng thấp càng tốt; Precision/Recall càng cao càng tốt.


In [ ]:
# Auto-generated from BIG_DATA_Customer_Intelligence_PRO_Colab.ipynb

# ---- source cell 52 ----
def popularity_recommend(train, users, k=10):
    pop=(train.assign(relevant=train.rating>=RELEVANCE_THRESHOLD)
         .groupby('item').agg(rel=('relevant','sum'),count=('rating','count'),avg=('rating','mean'))
         .sort_values(['rel','avg','count'],ascending=False).index.astype(int).tolist())
    seen=train.groupby('user').item.apply(set).to_dict(); out={}
    for u in users: out[u]=[it for it in pop if it not in seen.get(u,set())][:k]
    return out

def evaluate_models_once(seed):
    train,test=per_user_split(ratings_pd,seed,TEST_FRAC,MIN_USER_RATINGS_FOR_TEST)
    eval_users=test[test.rating>=RELEVANCE_THRESHOLD].user.unique().tolist()
    rating_rows=[]; ranking_rows=[]
    global_mean=float(train.rating.mean()); user_mean=train.groupby('user').rating.mean().to_dict(); item_mean=train.groupby('item').rating.mean().to_dict()
    baseline_preds={
        'GlobalMean':np.full(len(test),global_mean),
        'UserMean':np.array([user_mean.get(u,global_mean) for u in test.user]),
        'ItemMean':np.array([item_mean.get(i,global_mean) for i in test.item]),
    }
    for name,p in baseline_preds.items(): rating_rows.append({'seed':seed,'model':name,**rating_metrics(test.rating,p)})

    # UserCF
    ustate=build_usercf_state(train); up=usercf_predict(ustate,test,best_ucf)
    rating_rows.append({'seed':seed,'model':'UserCF',**rating_metrics(test.rating,up)})
    urec=usercf_recommend(ustate,eval_users,TOP_K,best_ucf)
    ranking_rows.append({'seed':seed,'model':'UserCF',**ranking_metrics(urec,test,TOP_K,RELEVANCE_THRESHOLD)})

    # SVD reconstruction
    sstate=build_svd_recommender(train,best_svd_rec_k); sp=svd_predict(sstate,test)
    rating_rows.append({'seed':seed,'model':'SVD-Reconstruction',**rating_metrics(test.rating,sp)})
    srec=svd_recommend(sstate,eval_users,TOP_K)
    ranking_rows.append({'seed':seed,'model':'SVD-Reconstruction',**ranking_metrics(srec,test,TOP_K,RELEVANCE_THRESHOLD)})

    # Popularity ranking baseline
    prec=popularity_recommend(train,eval_users,TOP_K)
    ranking_rows.append({'seed':seed,'model':'Popularity',**ranking_metrics(prec,test,TOP_K,RELEVANCE_THRESHOLD)})

    # ALS
    amodel=fit_als(train,best_als,seed); am=als_rating_eval(amodel,test)
    rating_rows.append({'seed':seed,'model':'ALS',**am})
    arec=als_recommend_dict(amodel,train,eval_users,TOP_K)
    ranking_rows.append({'seed':seed,'model':'ALS',**ranking_metrics(arec,test,TOP_K,RELEVANCE_THRESHOLD)})
    return rating_rows,ranking_rows,amodel,train,test,arec

all_rating=[]; all_ranking=[]; first_als_model=None; first_train=None; first_test=None; first_als_recs=None
for seed in STAT_SEEDS:
    rr,rk,amodel,tr,te,recs=evaluate_models_once(seed)
    all_rating.extend(rr); all_ranking.extend(rk)
    if first_als_model is None:
        first_als_model,first_train,first_test,first_als_recs=amodel,tr,te,recs

rating_runs=pd.DataFrame(all_rating); ranking_runs=pd.DataFrame(all_ranking)
rating_stats=rating_runs.groupby('model').agg(RMSE_mean=('RMSE','mean'),RMSE_std=('RMSE','std'),MAE_mean=('MAE','mean'),MAE_std=('MAE','std')).reset_index()
ranking_stats=ranking_runs.groupby('model').agg(Precision_mean=('Precision@K','mean'),Precision_std=('Precision@K','std'),Recall_mean=('Recall@K','mean'),Recall_std=('Recall@K','std'),eval_users=('eval_users','mean')).reset_index()

display(rating_stats.sort_values('RMSE_mean'))
display(ranking_stats.sort_values('Recall_mean',ascending=False))

# ---- source cell 54 ----
first_labels=(first_test[first_test.rating>=RELEVANCE_THRESHOLD].groupby('user').item.apply(list).to_dict())
rank_rows=[]
for u,labels in first_labels.items():
    if u in first_als_recs and labels:
        rank_rows.append(([float(x) for x in first_als_recs[u]],[float(x) for x in labels]))
if rank_rows:
    rank_eval_df=spark.createDataFrame(rank_rows,['prediction','label'])
    spark_precision=RankingEvaluator(predictionCol='prediction',labelCol='label',metricName='precisionAtK',k=TOP_K).evaluate(rank_eval_df)
    spark_recall=RankingEvaluator(predictionCol='prediction',labelCol='label',metricName='recallAtK',k=TOP_K).evaluate(rank_eval_df)
    manual_first=ranking_runs[(ranking_runs.seed==STAT_SEEDS[0])&(ranking_runs.model=='ALS')].iloc[0]
    print(f'Spark Precision@{TOP_K}={spark_precision:.6f} | manual={manual_first["Precision@K"]:.6f}')
    print(f'Spark Recall@{TOP_K}={spark_recall:.6f} | manual={manual_first["Recall@K"]:.6f}')
else:
    print('No users with held-out relevant items for RankingEvaluator cross-check.')

# ---- source cell 55 ----
# Automatic champion score for models that have both rating and ranking metrics.
model_score=rating_stats.merge(ranking_stats,on='model',how='inner')
for col in ['RMSE_mean','MAE_mean','Precision_mean','Recall_mean']:
    lo,hi=model_score[col].min(),model_score[col].max()
    model_score[col+'_n']=(model_score[col]-lo)/(hi-lo+1e-9)
model_score['ChampionScore']=(
    .25*(1-model_score['RMSE_mean_n']) + .25*(1-model_score['MAE_mean_n']) +
    .25*model_score['Precision_mean_n'] + .25*model_score['Recall_mean_n']
)
model_score=model_score.sort_values('ChampionScore',ascending=False)
champion=model_score.iloc[0]
display(model_score[['model','RMSE_mean','MAE_mean','Precision_mean','Recall_mean','ChampionScore']])
print('MODEL CHAMPION:',champion['model'])

# ---- source cell 57 ----
DEMO_USER = int(sorted(first_train.user.unique())[0])
champion_name = str(champion['model'])

def champion_recommendation_demo(user_id, k=10):
    seen=set(first_train[first_train.user==user_id].item.astype(int))
    if champion_name == 'ALS':
        users_sdf=spark.createDataFrame([(int(user_id),)],['user'])
        raw=first_als_model.recommendForUserSubset(users_sdf,min(len(set(first_train.item)),max(100,k*10)))
        rows=(raw.select('user',F.explode('recommendations').alias('rec'))
              .select(F.col('rec.item').alias('item'),F.col('rec.rating').alias('predicted_rating')).collect())
        data=[(int(r['item']),float(r['predicted_rating'])) for r in rows if int(r['item']) not in seen][:k]
        return pd.DataFrame(data,columns=['item','predicted_rating'])
    if champion_name == 'UserCF':
        st=build_usercf_state(first_train); rec=usercf_recommend(st,[user_id],k,best_ucf).get(user_id,[])
        pairs=pd.DataFrame({'user':[user_id]*len(rec),'item':rec})
        pairs['rating']=0.0
        pairs['predicted_rating']=usercf_predict(st,pairs,best_ucf) if len(pairs) else []
        return pairs[['item','predicted_rating']]
    if champion_name == 'SVD-Reconstruction':
        st=build_svd_recommender(first_train,best_svd_rec_k); rec=svd_recommend(st,[user_id],k).get(user_id,[])
        if not rec: return pd.DataFrame(columns=['item','predicted_rating'])
        ui=st['u2i'][user_id]
        return pd.DataFrame({'item':rec,'predicted_rating':[float(st['pred'][ui,st['i2i'][it]]) for it in rec]})
    return pd.DataFrame(columns=['item','predicted_rating'])

customer_demo_pd=champion_recommendation_demo(DEMO_USER,TOP_K)
customer_demo_pd.insert(0,'rank',np.arange(1,len(customer_demo_pd)+1))
print(f'Demo user={DEMO_USER} | champion={champion_name} | already-rated items={len(first_train[first_train.user==DEMO_USER])}')
display(customer_demo_pd)

if len(customer_demo_pd):
    fig_customer_demo=px.bar(
        customer_demo_pd.sort_values('rank',ascending=False),
        x='predicted_rating',y=customer_demo_pd.sort_values('rank',ascending=False)['item'].astype(str),orientation='h',
        hover_data={'rank':True,'predicted_rating':':.3f'},
        title=f'Customer-facing Top-{TOP_K} recommendations — user {DEMO_USER}'
    )
    fig_customer_demo.update_yaxes(title='Item')
    fig_customer_demo.show()

# ---- source cell 58 ----
fig_model_quality=make_subplots(rows=1,cols=2,subplot_titles=('Rating accuracy — lower is better','Top-N quality — higher is better'))
rs=rating_stats.sort_values('RMSE_mean')
fig_model_quality.add_trace(go.Bar(x=rs.model,y=rs.RMSE_mean,error_y=dict(type='data',array=rs.RMSE_std.fillna(0)),name='RMSE'),row=1,col=1)
fig_model_quality.add_trace(go.Bar(x=rs.model,y=rs.MAE_mean,error_y=dict(type='data',array=rs.MAE_std.fillna(0)),name='MAE'),row=1,col=1)
ks=ranking_stats.sort_values('Recall_mean',ascending=False)
fig_model_quality.add_trace(go.Bar(x=ks.model,y=ks.Precision_mean,error_y=dict(type='data',array=ks.Precision_std.fillna(0)),name='Precision@10'),row=1,col=2)
fig_model_quality.add_trace(go.Bar(x=ks.model,y=ks.Recall_mean,error_y=dict(type='data',array=ks.Recall_std.fillna(0)),name='Recall@10'),row=1,col=2)
fig_model_quality.update_layout(title='Recommendation benchmark — mean ± std across seeds',barmode='group',height=520)
fig_model_quality.show()


## 8. Spark Performance Lab

**Mục tiêu:** chứng minh vai trò của Spark bằng benchmark có kiểm soát: tăng scale, throughput, cache reuse và physical execution plan.

**Nguyên tắc:** nếu pandas nhanh hơn trên dữ liệu nhỏ/local thì notebook nói rõ overhead; không tạo claim sai để làm Spark trông tốt hơn.

**Output:** scaling curves, throughput, cache speedup và plan evidence.


In [ ]:
# Auto-generated from BIG_DATA_Customer_Intelligence_PRO_Colab.ipynb

# ---- source cell 60 ----
def spark_top_product_job(df):
    return df.groupBy('itemDescription').count().orderBy(F.desc('count')).limit(100).collect()

def benchmark_scaling(base_spark, scales=(1,10,25,50)):
    base_pd=base_spark.select('itemDescription').toPandas()
    out=[]
    for scale in scales:
        # Spark replication without collecting the large dataset to driver.
        big=(base_spark.select('itemDescription').crossJoin(spark.range(scale).select(F.col('id').alias('_rep'))).drop('_rep'))
        rows=base_spark.count()*scale
        t0=time.perf_counter(); spark_top_product_job(big); ts=time.perf_counter()-t0
        out.append({'scale':scale,'rows':rows,'engine':'Spark','runtime_sec':ts,'throughput_rows_sec':rows/max(ts,1e-9)})

        # Bounded pandas benchmark to avoid turning the demo into an OOM test.
        if rows <= 2_000_000:
            t0=time.perf_counter(); pbig=pd.concat([base_pd]*scale,ignore_index=True)
            _=pbig.groupby('itemDescription').size().nlargest(100)
            tp=time.perf_counter()-t0
            out.append({'scale':scale,'rows':rows,'engine':'pandas','runtime_sec':tp,'throughput_rows_sec':rows/max(tp,1e-9)})
            del pbig
    return pd.DataFrame(out)

SPARK_SCALES=(1,10,25,50) if QUALITY_MODE=='max' else (1,10,25)
scaling_pd=benchmark_scaling(baskets_df,SPARK_SCALES)
display(scaling_pd)

# ---- source cell 61 ----
fig_scaling=px.line(
    scaling_pd,x='rows',y='runtime_sec',color='engine',markers=True,
    hover_data={'scale':True,'throughput_rows_sec':':,.0f'},
    title='Compute scalability — runtime vs rows'
)
fig_scaling.update_xaxes(type='log',title='Rows (log scale)')
fig_scaling.update_yaxes(title='Runtime (seconds)')
fig_scaling.show()

fig_throughput=px.line(
    scaling_pd,x='rows',y='throughput_rows_sec',color='engine',markers=True,
    title='Compute scalability — throughput'
)
fig_throughput.update_xaxes(type='log',title='Rows (log scale)')
fig_throughput.update_yaxes(title='Rows / second')
fig_throughput.show()

# ---- source cell 62 ----
# Cache experiment on the largest scale.
cache_scale=max(SPARK_SCALES)
big_cache=(baskets_df.select('itemDescription').crossJoin(spark.range(cache_scale).select(F.col('id').alias('_rep'))).drop('_rep'))

t0=time.perf_counter(); spark_top_product_job(big_cache); uncached_sec=time.perf_counter()-t0
big_cache=big_cache.cache(); big_cache.count()  # materialize once
t0=time.perf_counter(); spark_top_product_job(big_cache); cached_sec=time.perf_counter()-t0
big_cache.unpersist()
cache_speedup=uncached_sec/max(cached_sec,1e-9)
print(f'Repeated-query cache: uncached={uncached_sec:.3f}s, cached={cached_sec:.3f}s, speedup={cache_speedup:.2f}x')

print('\nPhysical plan of a representative Spark aggregation:')
baskets_df.groupBy('itemDescription').count().orderBy(F.desc('count')).limit(10).explain('formatted')


## 9. Automatic conclusions, Executive Dashboard & exports

**Mục tiêu:** tổng hợp toàn bộ evidence thành Executive Dashboard, automatic conclusions và artifacts có thể giao cho khách hàng.

**Output:** KPI cards có tooltip, model/cluster/rule/scalability charts, CSV, HTML dashboard, model artifacts và ZIP export.

**Cách demo:** bắt đầu từ dashboard/kết luận rồi mới drill-down về code và metric phía trên.


In [ ]:
# Auto-generated from BIG_DATA_Customer_Intelligence_PRO_Colab.ipynb

# ---- source cell 64 ----
# Compute evidence-based narratives.
conclusions=[]
conclusions.append(
    f"SVD selected {selected_svd_k} latent dimensions, capturing {cum_energy[selected_svd_k-1]:.1%} of centered-matrix energy."
)
conclusions.append(
    f"CURE selected k={int(best_cure['k'])}, representatives={int(best_cure['representatives'])}, compression={best_cure['compression']:.2f}; "
    f"final silhouette={cure_sil:.3f}, stability ARI={cure_stability_ari:.3f}."
)
conclusions.append(
    f"Recommendation champion = {champion['model']} with RMSE={champion['RMSE_mean']:.3f}, MAE={champion['MAE_mean']:.3f}, "
    f"Precision@{TOP_K}={champion['Precision_mean']:.3f}, Recall@{TOP_K}={champion['Recall_mean']:.3f}."
)
if len(fp_rules_pd):
    br=fp_rules_pd.iloc[0]
    conclusions.append(
        f"Top actionable market-basket rule: {br['antecedent_text']} → {br['consequent_text']} "
        f"(confidence={br['confidence']:.3f}, lift={br['lift']:.3f}, support={br['support']:.4f})."
    )

common=scaling_pd.groupby('rows').filter(lambda g: set(g.engine)=={'Spark','pandas'})
if len(common):
    largest=common.rows.max(); comp=common[common.rows==largest].set_index('engine')
    if comp.loc['Spark','runtime_sec'] < comp.loc['pandas','runtime_sec']:
        conclusions.append(f"At {int(largest):,} rows in this local benchmark, Spark is faster than pandas for the tested aggregation.")
    else:
        conclusions.append(f"At {int(largest):,} rows in this local benchmark, pandas remains faster; Spark's demonstrated value here is distributed execution semantics, scaling path, physical-plan visibility and cache reuse—not a fabricated single-node speed claim.")
conclusions.append(f"Spark cache speedup for the repeated aggregation at scale {cache_scale}× = {cache_speedup:.2f}×.")

for i,c in enumerate(conclusions,1): print(f'{i}. {c}')

# ---- source cell 66 ----
def fmt_num(x,kind='num'):
    if kind=='pct': return f'{x:.1%}'
    if isinstance(x,(float,np.floating)): return f'{x:.3f}'
    return f'{int(x):,}' if isinstance(x,(int,np.integer)) else str(x)

def kpi_card(label,value,tooltip):
    return f"""
    <div title="{tooltip}" style='background:white;border:1px solid #E2E8F0;border-radius:14px;padding:14px 16px;min-width:155px;box-shadow:0 2px 10px rgba(15,23,42,.05)'>
      <div style='font-size:12px;color:#64748B;font-weight:600'>{label} ⓘ</div>
      <div style='font-size:24px;color:#0F172A;font-weight:750;margin-top:4px'>{value}</div>
    </div>"""

cards=''.join([
    kpi_card('Ratings',fmt_num(ratings_rows),'Observed user-item ratings.'),
    kpi_card('Users',fmt_num(n_users),'Unique users in ratings2k.'),
    kpi_card('Sparsity',fmt_num(sparsity,'pct'),'1 - observed ratings / all possible user-item pairs.'),
    kpi_card('Model champion',str(champion['model']),'Balanced score across RMSE, MAE, Precision@10 and Recall@10.'),
    kpi_card('RMSE',fmt_num(champion['RMSE_mean']),'Lower is better. Mean across statistical-validation seeds.'),
    kpi_card(f'Precision@{TOP_K}',fmt_num(champion['Precision_mean']),'Relevant recommendations divided by K.'),
    kpi_card(f'Recall@{TOP_K}',fmt_num(champion['Recall_mean']),'Share of held-out relevant items recovered in Top-K.'),
    kpi_card('SVD dimensions',fmt_num(selected_svd_k),f'Chosen by captured energy; {cum_energy[selected_svd_k-1]:.1%} captured.'),
    kpi_card('CURE k',fmt_num(int(best_cure['k'])),'Selected by silhouette, DB, CH and balance composite.'),
    kpi_card('CURE silhouette',fmt_num(cure_sil),'Higher indicates stronger separation/cohesion in latent space.'),
    kpi_card('CURE stability ARI',fmt_num(cure_stability_ari),'Agreement of cluster assignments across repeated samples; 1 is perfect.'),
    kpi_card('Spark cache speedup',f'{cache_speedup:.2f}×','Repeated aggregation after cache materialization vs uncached run.'),
])

dashboard_html=f"""
<div style='font-family:Inter,Arial,sans-serif;background:#F8FAFC;border-radius:18px;padding:20px;border:1px solid #E2E8F0'>
  <div style='display:flex;justify-content:space-between;align-items:end;margin-bottom:14px'>
    <div><div style='font-size:12px;color:#2563EB;font-weight:700;letter-spacing:.08em'>EXECUTIVE VIEW</div>
    <div style='font-size:25px;font-weight:800;color:#0F172A'>Big Data Customer Intelligence</div></div>
    <div style='font-size:12px;color:#64748B'>Hover KPI titles / charts for evidence</div>
  </div>
  <div style='display:flex;gap:10px;flex-wrap:wrap'>{cards}</div>
</div>
"""

display(HTML(dashboard_html))
fig_model_quality.show()
fig_cluster_umap.show()
fig_rules.show()
fig_scaling.show()

if EXEC_OUT is not None:
    try:
        with EXEC_OUT:
            clear_output(wait=True)
            display(HTML(dashboard_html))
            display(fig_model_quality)
            display(fig_cluster_umap)
            display(fig_rules)
            display(fig_scaling)
    except Exception as e:
        print('Top dashboard refresh skipped:',e)

# ---- source cell 68 ----
# Tables
rating_stats.to_csv(OUTPUT_DIR/'recommendation_rating_metrics.csv',index=False)
ranking_stats.to_csv(OUTPUT_DIR/'recommendation_ranking_metrics.csv',index=False)
model_score.to_csv(OUTPUT_DIR/'recommendation_model_champion.csv',index=False)
cure_tuning_pd.to_csv(OUTPUT_DIR/'cure_tuning.csv',index=False)
cluster_profile.to_csv(OUTPUT_DIR/'cluster_profiles.csv',index=False)
rfv_pd.to_csv(OUTPUT_DIR/'rfv_customer_segments.csv',index=False)
fp_tuning_pd.to_csv(OUTPUT_DIR/'fpgrowth_tuning.csv',index=False)
fp_rules_pd.to_csv(OUTPUT_DIR/'fpgrowth_top_rules.csv',index=False)
scaling_pd.to_csv(OUTPUT_DIR/'spark_scalability.csv',index=False)
svd_curve.to_csv(OUTPUT_DIR/'svd_energy_curve.csv',index=False)

# CURE representative points
rep_rows=[]
for cid,reps in enumerate(cure_reps):
    for rid,p in enumerate(reps):
        rep_rows.append({'cluster':cid,'representative':rid,**{f'z{i+1}':float(v) for i,v in enumerate(p)}})
pd.DataFrame(rep_rows).to_csv(OUTPUT_DIR/'cure_representatives.csv',index=False)

# Spark ALS model from first statistical-validation seed.
try:
    first_als_model.write().overwrite().save(str(OUTPUT_DIR/'als_model'))
except Exception as e:
    print('ALS export warning:',e)

# One HTML executive report.
parts=["<html><head><meta charset='utf-8'><title>Big Data Executive Dashboard</title></head><body style='font-family:Arial;background:#f8fafc;padding:24px'>",dashboard_html]
figures=[fig_model_quality,fig_cluster_umap,fig_cluster_radar,fig_rules,fig_rule_network,fig_rfv,fig_svd,fig_scaling,fig_throughput]
for i,fig in enumerate(figures):
    parts.append(pio.to_html(fig,full_html=False,include_plotlyjs='cdn' if i==0 else False))
parts.append('<h2>Automatic conclusions</h2><ol>'+''.join(f'<li>{c}</li>' for c in conclusions)+'</ol></body></html>')
(OUTPUT_DIR/'executive_dashboard.html').write_text('\n'.join(parts),encoding='utf-8')

# Save conclusions JSON for reproducibility.
(OUTPUT_DIR/'summary.json').write_text(json.dumps({
    'data_kpis':DATA_KPIS,
    'selected_svd_k':selected_svd_k,
    'cure':{k:(float(v) if isinstance(v,(np.floating,float)) else int(v) if isinstance(v,(np.integer,int)) else v) for k,v in best_cure.items() if k in ['k','representatives','compression','silhouette','davies_bouldin','calinski_harabasz']},
    'champion':{k:(float(v) if isinstance(v,(np.floating,float)) else v) for k,v in champion.to_dict().items()},
    'conclusions':conclusions
},indent=2,ensure_ascii=False),encoding='utf-8')

zip_path=shutil.make_archive('/content/BIG_DATA_Customer_Intelligence_Outputs','zip',OUTPUT_DIR)
print('Exported:',zip_path)
print('HTML dashboard:',OUTPUT_DIR/'executive_dashboard.html')
print('All artifacts:',sorted(p.name for p in OUTPUT_DIR.iterdir()))

# ---- source cell 69 ----
# Optional Colab downloads — uncomment when needed.
# from google.colab import files
# files.download('/content/BIG_DATA_Customer_Intelligence_Outputs.zip')
# files.download('/content/big_data_outputs/executive_dashboard.html')


## What to show first
1. Executive Dashboard / model champion.
2. FP-Growth cross-sell evidence.
3. RFV customer segments.
4. SVD latent-space evidence.
5. CURE UMAP + cluster personas/stability.
6. Recommendation RMSE/MAE + Precision@K/Recall@K.
7. Spark scalability/cache benchmark.
8. Automatic conclusions and exported HTML/ZIP artifacts.

**Interpretation rule:** nếu Spark không thắng ở dữ liệu demo nhỏ, notebook giải thích overhead thay vì bóp méo benchmark.